In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import sys
sys.path.append("../..")  # Add repository root to Python path

import torch
import torch.nn as nn
from convlstm.fireseq2seq import FireSeq2Seq

In [3]:
dataset = xr.open_dataset("../../combined_data/jan2025.nc")
dataset

<xarray.Dataset> Size: 3MB
Dimensions:     (valid_time: 744, latitude: 8, longitude: 16)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 6kB 2025-01-01 ... 2025-01-31T23:...
  * latitude    (latitude) float64 64B 34.75 34.65 34.55 ... 34.25 34.15 34.05
  * longitude   (longitude) float64 128B -119.0 -118.9 -118.8 ... -117.6 -117.5
    expver      (valid_time) <U4 12kB ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 381kB ...
    u10         (valid_time, latitude, longitude) float32 381kB ...
    v10         (valid_time, latitude, longitude) float32 381kB ...
    tp          (valid_time, latitude, longitude) float32 381kB ...
    lai_hv      (valid_time, latitude, longitude) float32 381kB ...
    lai_lv      (valid_time, latitude, longitude) float32 381kB ...
    frp         (valid_time, latitude, longitude) float32 381kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-03-18T21:20 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
frp = torch.tensor(dataset['frp'].values, dtype=torch.float32)
fire_mask = (frp > 0).int()
frp.shape, fire_mask.shape

(torch.Size([744, 8, 16]), torch.Size([744, 8, 16]))

In [5]:
fire_mask.sum()

tensor(419)

In [6]:
fire_mask.isnan().sum()

tensor(0)

In [7]:
y = fire_mask

In [8]:
input_vars = ["d2m", "u10", "v10", "tp", "lai_hv", "lai_lv"]
X = torch.stack([torch.tensor(dataset[input_var].values, dtype=torch.float32) for input_var in input_vars], dim=1)
X.shape

torch.Size([744, 6, 8, 16])

In [9]:
# Check for NaN values in the input data
print(f"Number of NaN values in X: {torch.isnan(X).sum().item()}")

# Function to interpolate NaN values with the average of neighboring values
def interpolate_nans(tensor):
    # Create a copy of the tensor to avoid modifying the original
    interpolated = tensor.clone()
    
    # Get the dimensions
    time_steps, features, height, width = tensor.shape
    
    # Iterate through each time step
    for t in range(time_steps):
        # Iterate through each feature
        for f in range(features):
            # Find positions with NaN values
            nan_mask = torch.isnan(tensor[t, f, :, :])
            if not torch.any(nan_mask):
                continue  # Skip if no NaNs in this feature at this time step
            
            # For each NaN position, compute the average of non-NaN neighbors
            for h in range(height):
                for w in range(width):
                    if nan_mask[h, w]:
                        # Define neighbor indices (considering boundaries)
                        h_start = max(0, h-1)
                        h_end = min(height, h+2)
                        w_start = max(0, w-1)
                        w_end = min(width, w+2)
                        
                        # Extract the neighborhoobd
                        neighborhood = tensor[t, f, h_start:h_end, w_start:w_end]
                        # Calculate mean of non-NaN neighbors
                        valid_neighbors = neighborhood[~torch.isnan(neighborhood)]
                        
                        if len(valid_neighbors) > 0:
                            # Replace NaN with mean of valid neighbors
                            interpolated[t, f, h, w] = valid_neighbors.mean()
                        else:
                            # If all neighbors are NaN, use global mean for this feature
                            feature_data = tensor[:, f, :, :]
                            global_mean = feature_data[~torch.isnan(feature_data)].mean()
                            interpolated[t, f, h, w] = global_mean
    
    return interpolated

# Apply the interpolation
X = interpolate_nans(X)

# Verify that all NaN values have been interpolated
print(f"Number of NaN values after interpolation: {torch.isnan(X).sum().item()}")

Number of NaN values in X: 4464
Number of NaN values after interpolation: 0


In [10]:
# time_steps, height, width, features = X.shape
time_steps, features, height, width = X.shape

In [11]:
# Normalize all features in X
# We'll normalize each feature independently across all time steps, locations

# Create a copy of X to store normalized values
X_normalized = X.clone()

# Normalize each feature, exclude wind direction since that's already normalized, in a different way
for f in range(features):
    # Extract the feature across all time steps and locations
    feature_data = X[:, f, :, :]
    
    # Calculate mean and standard deviation of non-NaN values
    feature_mean = feature_data.mean()
    feature_std = feature_data.std()
    
    # Normalize the feature (z-score normalization)
    X_normalized[:, f, :, :] = (feature_data - feature_mean) / feature_std

    
    print(f"Feature {f} - Mean: {feature_mean:.4f}, Std: {feature_std:.4f}")

# Replace the original X with the normalized version
X = X_normalized

print(f"Data normalized. New feature ranges:")
for f in range(features):
    feature_min = X[:, f, :, :].min().item()
    feature_max = X[:, f, :, :].max().item()
    print(f"Feature {f} - Min: {feature_min:.4f}, Max: {feature_max:.4f}")

Feature 0 - Mean: 266.8738, Std: 8.2098
Feature 1 - Mean: -0.5615, Std: 1.5268
Feature 2 - Mean: -1.0642, Std: 1.8361
Feature 3 - Mean: 0.0003, Std: 0.0016
Feature 4 - Mean: 3.1635, Std: 1.6111
Feature 5 - Mean: 1.7614, Std: 0.8092
Data normalized. New feature ranges:
Feature 0 - Min: -2.4544, Max: 2.1073
Feature 1 - Min: -5.2942, Max: 5.5459
Feature 2 - Min: -4.7070, Max: 4.0335
Feature 3 - Min: -0.2122, Max: 9.8967
Feature 4 - Min: -1.9636, Max: 1.1238
Feature 5 - Min: -1.5395, Max: 1.2822


In [12]:
from torch.utils.data import Dataset, DataLoader, Subset

class FireDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [13]:
def create_sequences(data, seq_length):
    sequences = []
    targets = []
    for i in range(len(data) - seq_length):
        # Extract sequence
        seq = data[i:i+seq_length]
        # Target is the next time step after the sequence
        target = y[i:i+seq_length]
        sequences.append(seq)
        targets.append(target)
    return torch.stack(sequences), torch.stack(targets)

In [32]:
def mse_nan_loss(output, target):
    # mask = ~torch.isnan(target)
    mask = (target != 0)
    sqe = (target[mask] - output[mask])**2
    return sqe.mean()

In [ ]:
def train_model(model, train_loader, val_loader, test_loader, optimizer, num_epochs, device="cuda"):
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    mse_loss = torch.nn.MSELoss()
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            # loss = mse_nan_loss(outputs, targets)
            loss = mse_loss(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
        
        train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                
                outputs = model(inputs)
                # loss = mse_nan_loss(outputs, targets)
                loss = mse_loss(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
        
        val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(val_loss)
        print(f'Epoch {epoch+1}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f}')
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "./old_dataset/best_model.pth")
    
    # Final test evaluation
    model.load_state_dict(torch.load("./old_dataset/best_model.pth"))
    model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            # loss = mse_nan_loss(outputs, targets)
            loss = mse_loss(outputs, targets)
            test_loss += loss.item() * inputs.size(0)
    
    test_loss = test_loss / len(test_loader.dataset)
    print(f'\nFinal Test Loss: {test_loss:.4f}')
    
    # Plot and save the training curves
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss Over Time')
    plt.legend()
    plt.grid(True)
    
    # Save the plot
    plt.savefig('./old_dataset/training_validation_loss.png')
    plt.close()
    
    return model

In [34]:
seq_length = 12
X_seq, y_seq = create_sequences(X, seq_length)
X_seq = torch.permute(X_seq, (0, 2, 1, 3, 4))
y_seq = torch.unsqueeze(y_seq, 1).float()

In [35]:
X_seq.shape, y_seq.shape

(torch.Size([732, 6, 12, 8, 16]), torch.Size([732, 1, 12, 8, 16]))

In [36]:
y_seq = y_seq.float()

In [37]:
# Create full dataset
full_dataset = FireDataset(X_seq, y_seq)

# Calculate split indices
total_size = len(full_dataset)
train_end = int(0.7 * total_size)
val_end = train_end + int(0.1 * total_size)

# Create sequential splits
train_indices = range(0, train_end)
val_indices = range(train_end, val_end)
test_indices = range(val_end, total_size)

# Create subsets
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

# Create DataLoaders without shuffling
batch_size = 24
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [38]:
train_indices, val_indices, test_indices

(range(0, 512), range(512, 585), range(585, 732))

In [39]:
train_dataset.indices, val_dataset.indices, test_dataset.indices

(range(0, 512), range(512, 585), range(585, 732))

In [40]:
# Define model parameters
# height = X.shape[1]
height = X.shape[2]
# width = X.shape[2]
width = X.shape[3]
# in_channels = X.shape[3]
in_channels = X.shape[1]
out_channels = 1 # just 1 for frp/fire_mask

X.shape

torch.Size([744, 6, 8, 16])

In [41]:
frame_size = (height, width)
kernel_size = (3, 3)
num_layers = 1
bias = True

# Create the model
model = FireSeq2Seq(
    in_channels = in_channels,
    out_channels = out_channels,
    frame_size = frame_size,
    kernel_size = kernel_size,
    num_layers = num_layers,
    bias = bias
).to("cuda")

model

FireSeq2Seq(
  (sequential): Sequential(
    (ConvLSTM_1): ConvLSTM(
      (cell): ConvLSTMCell(
        (conv): Conv2d(70, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
    )
    (BatchNorm_1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv): Conv2d(64, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

In [42]:
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
mse_loss = torch.nn.MSELoss()
# Training parameters
num_epochs = 10
device = "cuda"

In [43]:
trained_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    num_epochs=num_epochs,
    device="cuda"
)

Epoch 1/10 | Train Loss: 0.1200 | Val Loss: 0.0092
Epoch 2/10 | Train Loss: 0.0372 | Val Loss: 0.0117
Epoch 3/10 | Train Loss: 0.0234 | Val Loss: 0.0191
Epoch 4/10 | Train Loss: 0.0175 | Val Loss: 0.0198
Epoch 5/10 | Train Loss: 0.0144 | Val Loss: 0.0175
Epoch 6/10 | Train Loss: 0.0127 | Val Loss: 0.0158
Epoch 7/10 | Train Loss: 0.0116 | Val Loss: 0.0147
Epoch 8/10 | Train Loss: 0.0108 | Val Loss: 0.0139
Epoch 9/10 | Train Loss: 0.0102 | Val Loss: 0.0132
Epoch 10/10 | Train Loss: 0.0097 | Val Loss: 0.0125

Final Test Loss: 0.0059


In [3]:
def visualize_sequence(model, dataloader, sample_idx=0, threshold=0.5):
    inputs, targets = next(iter(dataloader))
    inputs = inputs.to(device)
    targets = targets.to(device)
    
    model.eval()
    with torch.no_grad():
        predictions = model(inputs)
    
    pred = predictions[sample_idx].cpu().numpy()
    true = targets[sample_idx].cpu().numpy()
    pred_binary = (pred > threshold).astype(np.float32)
    
    seq_length = pred.shape[1]
    fig, axes = plt.subplots(2, seq_length, figsize=(5*seq_length, 10))
    
    for t in range(seq_length):
        # Prediction
        axes[0,t].imshow(pred_binary[0,t], cmap='binary')
        axes[0,t].set_title(f'Pred t={t}')
        
        # Ground truth
        axes[1,t].imshow(true[0,t], cmap='binary')
        axes[1,t].set_title(f'True t={t}')
    
    plt.tight_layout()
    plt.savefig('sequence_comparison.png', dpi=300)
    plt.show()

In [ ]:
def calculate_metrics(model, dataloader, threshold=0.5):
    model.eval()
    total_tp = total_fp = total_fn = total_tn = 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            preds = model(inputs)
            preds_binary = (preds > threshold).float()
            
            # Flatten all dimensions except batch
            pred_flat = preds_binary.flatten(start_dim=1)
            target_flat = targets.flatten(start_dim=1)
            
            # Calculate confusion matrix
            tp = (pred_flat * target_flat).sum(dim=1)  # True positives
            fp = (pred_flat * (1-target_flat)).sum(dim=1)  # False positives
            fn = ((1-pred_flat) * target_flat).sum(dim=1)  # False negatives
            tn = ((1-pred_flat) * (1-target_flat)).sum(dim=1)  # True negatives
            
            total_tp += tp.sum().item()
            total_fp += fp.sum().item()
            total_fn += fn.sum().item()
            total_tn += tn.sum().item()
    
    # Calculate metrics
    precision = total_tp / (total_tp + total_fp + 1e-10)
    recall = total_tp / (total_tp + total_fn + 1e-10)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    accuracy = (total_tp + total_tn) / (total_tp + total_tn + total_fp + total_fn)
    
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Confusion Matrix:')
    print(f'[[{total_tn} {total_fp}]')
    print(f' [{total_fn} {total_tp}]]')
    
    return {'precision': precision, 'recall': recall, 'f1': f1, 'accuracy': accuracy}

In [4]:
model = torch.load("./ee_dataset/best_model.pth")
# Visualize
visualize_sequence(model, train_loader)

# Get metrics
metrics = calculate_metrics(model, train_loader)

NameError: name 'torch' is not defined